# Fill Period Err from NASA Cumulative Table

Reads `final_vetting.csv`, extracts KIC IDs from the `Target` column, queries NASA's `cumulative` table for `koi_period_err1` (official Kepler pipeline period uncertainty), and writes back filled values into `Period Err (d)`.

In [ ]:
!pip install -q pandas requests

In [ ]:
import time
import numpy as np
import pandas as pd
import requests
from io import StringIO

INPUT  = 'final_final_vetting.csv'
OUTPUT = 'final_final_vetting.csv'   # overwrite in place; change to save a copy

TAP_URL = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'
BATCH_SIZE = 300

df = pd.read_csv(INPUT)
print(f'Loaded {len(df)} rows from {INPUT}')
print(f'Period Err (d) missing: {df["Period Err (d)"].isna().sum()}')
df.head(3)

In [ ]:
# Extract KIC IDs from Target column
kic_ids = df['Target'].str.extract(r'(\d+)')[0].astype(int).tolist()
print(f'Extracted {len(kic_ids)} KIC IDs')

# Batch query NASA cumulative table for koi_period_err1
parts = []
for b in range(0, len(kic_ids), BATCH_SIZE):
    batch = kic_ids[b:b+BATCH_SIZE]
    ids_str = ','.join(str(x) for x in batch)
    q = f'select kepid, koi_period_err1 from cumulative where kepid in ({ids_str})'
    try:
        r = requests.get(TAP_URL, params={'query': q, 'format': 'csv'}, timeout=60)
        if r.status_code == 200 and 'kepid' in r.text:
            parts.append(pd.read_csv(StringIO(r.text)))
    except Exception as e:
        print(f'  Batch {b//BATCH_SIZE} failed: {e}')
    time.sleep(0.3)

if parts:
    archive = pd.concat(parts, ignore_index=True)
    archive = archive.drop_duplicates(subset=['kepid'])
    print(f'Retrieved {len(archive)} records from NASA')
else:
    raise RuntimeError('All NASA queries failed.')

In [ ]:
# Merge and fill
# Add temporary kepid column for joining
df['_kepid'] = df['Target'].str.extract(r'(\d+)')[0].astype(int)

before = df['Period Err (d)'].isna().sum()
df = df.merge(archive, left_on='_kepid', right_on='kepid', how='left')

# Fill: use NASA value where Period Err is missing
df['Period Err (d)'] = df.apply(
    lambda r: round(r['koi_period_err1'], 5)
    if pd.isna(r['Period Err (d)']) and pd.notna(r['koi_period_err1'])
    else r['Period Err (d)'], axis=1)

after = df['Period Err (d)'].isna().sum()

# Clean up temp columns
df = df.drop(columns=['_kepid', 'kepid', 'koi_period_err1'], errors='ignore')

# Save
df.to_csv(OUTPUT, index=False, encoding='utf-8-sig')
print(f'Done. Filled: {before - after} rows. Still missing: {after}')
print(f'Saved -> {OUTPUT}')
df.head(10)